# 01 â€” NER Walkthrough

Step-by-step demonstration of the Named Entity Recognition pipeline.

**What this notebook covers**
- Load and run the scispaCy NER model
- Inspect extracted entities
- Entity frequency analysis
- Visualise entities in context
- Batch processing on MTSamples

## 1. Setup

Make sure scispaCy is installed:
```bash
pip install spacy scispacy
pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.3/en_core_sci_lg-0.5.3.tar.gz
```

In [2]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import plotly.express as px

from src.nlp.ner import build_ner_pipeline
from src.utils.text_utils import clean_clinical_text

print('Imports OK')

Imports OK


## 2. Load the NER Pipeline

The model loads lazily on first call (~2â€“3 seconds).

In [3]:
ner = build_ner_pipeline()
print(f'Pipeline ready: {ner.model_name}')

Pipeline ready: hybrid(en_ner_bc5cdr_md + en_core_sci_lg)


## 3. Extract Entities from a Single Note

In [4]:
sample_note = """
Patient is a 58-year-old male presenting with acute chest pain
radiating to the left arm. History of hypertension, type 2 diabetes
mellitus, and hyperlipidaemia. Current medications: metformin 500mg
bid, atorvastatin 40mg, lisinopril 10mg. ECG shows ST elevation.
Impression: STEMI. Plan: urgent PCI.
"""

cleaned   = clean_clinical_text(sample_note)
entities  = ner.extract(cleaned)

print(f'Entities found: {len(entities)}')
print()
for ent in entities:
    print(f'  [{ent.label:12s}]  {ent.text}')

2026-06-26 10:59:34 | INFO     | src.nlp.ner                    | Loading NER model: en_ner_bc5cdr_md
2026-06-26 11:00:01 | INFO     | src.nlp.ner                    | NER model loaded ✓
2026-06-26 11:00:02 | INFO     | src.nlp.ner                    | Loading NER model: en_core_sci_lg
2026-06-26 11:00:27 | INFO     | src.nlp.ner                    | NER model loaded ✓
Entities found: 12

  [SYMPTOM     ]  chest pain
  [ANATOMY     ]  left arm
  [DISEASE     ]  hypertension
  [DISEASE     ]  type 2 diabetes
  [DISEASE     ]  hyperlipidaemia
  [MEDICATION  ]  metformin
  [MEDICATION  ]  atorvastatin
  [MEDICATION  ]  lisinopril
  [PROCEDURE   ]  ECG
  [SYMPTOM     ]  elevation
  [DISEASE     ]  STEMI
  [PROCEDURE   ]  PCI


## 4. Entity Frequency on a Sample Corpus

In [5]:
from src.etl.extract import load_mtsamples
from src.utils.text_utils import clean_clinical_text
from collections import Counter

df = load_mtsamples()

# Use a small sample to keep runtime reasonable in the notebook
sample_df = df.dropna(subset=['transcription']).head(200)

all_entities = []
for text in sample_df['transcription']:
    cleaned = clean_clinical_text(text)
    ents    = ner.extract(cleaned)
    all_entities.extend(ents)

print(f'Total entities extracted: {len(all_entities)}')
print(f'Unique texts:             {len(set(e.text for e in all_entities))}')

2026-06-26 11:00:27 | INFO     | src.etl.extract                | Loading MTSamples from C:\Users\Hp\Documents\clinical-nlp-pipeline\data\raw\mtsamples.csv
2026-06-26 11:00:28 | INFO     | src.etl.extract                | MTSamples loaded: 4999 notes, 40 specialties
Total entities extracted: 18876
Unique texts:             6889


In [6]:
# Count by label
by_label = Counter(e.label for e in all_entities)
for label, count in sorted(by_label.items(), key=lambda x: -x[1]):
    print(f'  {label:<12} {count:>5}')

  SYMPTOM      12494
  ANATOMY       3199
  PROCEDURE     1771
  DISEASE        960
  MEDICATION     452


In [7]:
# Top 15 diseases
diseases = [e.text.lower() for e in all_entities if e.label == 'DISEASE']
top_diseases = Counter(diseases).most_common(15)

fig = px.bar(
    x=[c for _, c in top_diseases],
    y=[t for t, _ in top_diseases],
    orientation='h',
    title='Top 15 disease mentions in sample corpus',
    labels={'x': 'Count', 'y': ''},
)
fig.update_layout(yaxis={'autorange': 'reversed'}, height=400)
fig.show()

## 5. Batch Processing

In [8]:
# Batch processing is more efficient than calling extract() in a loop
texts  = [clean_clinical_text(t) for t in sample_df['transcription'].head(50)]
result = ner.extract_batch(texts, batch_size=16)

total = sum(len(ents) for ents in result)
print(f'Processed {len(result)} notes, {total} entities total')
print(f'Avg entities per note: {total / len(result):.1f}')

Processed 50 notes, 4326 entities total
Avg entities per note: 86.5


## 6. Gold-Standard Precision Evaluation

Every other check in this notebook validated NER output by eyeballing
charts. That catches obvious problems but gives no reproducible number.
This section does the opposite: a stratified sample of 125 entities
(25 per label) extracted from 15 notes across 5 specialties was
**manually reviewed against the source note text** and labelled
correct/incorrect by hand.

**Caveat**: this was a self-review by the same person who wrote the
classification rules, not an independent clinical annotator -- treat
this as a sanity-check metric, not a publishable benchmark. The full
judgment data is in data/processed/ner_gold_standard.csv for
inspection.


In [9]:
import json
with open('../data/processed/ner_gold_standard_summary.json', encoding='utf-8') as f:
    gold_summary = json.load(f)

labels = [l for l in gold_summary if l != 'OVERALL']
precisions = [gold_summary[l]['precision'] for l in labels]

fig = px.bar(
    x=labels, y=precisions,
    title='NER label precision on hand-reviewed gold-standard sample (n=125)',
    labels={'x': '', 'y': 'Precision'},
    text=[f'{p:.0%}' for p in precisions],
    height=400,
)
fig.update_traces(textposition='outside')
fig.update_layout(yaxis_range=[0, 1.1], xaxis_title='', yaxis_title='Precision')
fig.show()

overall = gold_summary['OVERALL']
print(f'Overall: {overall["correct"]}/{overall["total"]} = {overall["precision"]:.1%}')


Overall: 92/125 = 73.6%


**Reading this chart**: PROCEDURE and DISEASE are the most reliable
categories. SYMPTOM is structurally the weakest — it is the default
bucket for any entity that does not match a more specific rule, so it
absorbs generic verbs, connector phrases, and equipment names that the
underlying NER models extract but that are not clinically meaningful
on their own. Improving SYMPTOM precision further would need either a
trained classifier or much more aggressive part-of-speech filtering —
see the Known Limitations section in notebook 04 for details.